# CuPyCCx — CCD Computational Scaling Study

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varunrishi/CuPyCCx/blob/master/examples/colab_scaling.ipynb)

**Before running:** set runtime to GPU via *Runtime → Change runtime type → T4 GPU*

CCD (Coupled Cluster Doubles) has a formal computational cost of **O(N²_occ · N⁴_vir)**,
dominated by the T₂ · W_vvvv contraction. This notebook empirically measures that scaling
using a hydrogen chain series H₂ → H₄ → H₈ → H₁₆ → H₃₂ → H₆₄ (atom count doubling each step)
in the STO-3G basis, where n_occ = n_vir = N (spin-orbitals) exactly.

Two backends are compared:
- **C++ / cuBLAS**: custom DGEMM-dispatched residual builder (`cupyccx.method.CCD`)
- **PyCCD / CuPy**: pure-Python einsum solver running on the GPU (`cupyccx.py_solver.PyCCD`)

Timing uses 5 fixed iterations per system with convergence checks disabled,
so the measured time reflects pure residual-contraction cost.

In [ ]:
# Cell 1 — confirm GPU is available
!nvidia-smi
!nvcc --version

In [ ]:
%%bash
# Cell 2 — install dependencies; remove any stale cupyccx install
apt-get install -qq cmake ninja-build libeigen3-dev libopenblas-dev
pip install -q --upgrade pip
pip install -q pybind11 pyscf
pip uninstall -q -y cupyccx 2>/dev/null || true

In [ ]:
%%bash
# Cell 3 — clone and build with CUDA (T4 = sm_75; A100 = sm_80)
# Always start from /content so re-running doesn't nest CuPyCCx/CuPyCCx/...
cd /content

rm -rf CuPyCCx
git clone --quiet https://github.com/varunrishi/CuPyCCx.git
cd CuPyCCx

# Use the system cmake from apt (/usr/bin/cmake), NOT the pip-installed cmake
# which cannot locate nvcc. pip install pyscf pulls in cmake 3.31 as a
# Python package and it shadows the system cmake on PATH.
/usr/bin/cmake -B build \
  -DCUPYCCX_CUDA=ON \
  -DCUPYCCX_CUDA_ARCH=75 \
  -DCMAKE_BUILD_TYPE=Release \
  -DCUPYCCX_BUILD_TESTS=OFF \
  -Dpybind11_DIR=$(python3 -c "import pybind11; print(pybind11.get_cmake_dir())")
/usr/bin/cmake --build build -j$(nproc)

# Install Python files + CUDA extension directly into site-packages
SITE=$(python3 -c "import site; print(site.getsitepackages()[0])")
rm -rf "$SITE/cupyccx"
cp -r python/cupyccx "$SITE/"
cp build/_cupyccx*.so "$SITE/cupyccx/"
echo "Installed to: $SITE/cupyccx/"
ls "$SITE/cupyccx/"

# Verify import works before leaving bash
python3 -c "import importlib; importlib.invalidate_caches(); import cupyccx._cupyccx; print('Extension OK:', cupyccx._cupyccx.__file__)"

In [ ]:
# Cell 4 — verify extension loads in the notebook kernel
import sys, importlib, os, glob

# Evict all stale cupyccx modules
for key in list(sys.modules):
    if 'cupyccx' in key:
        del sys.modules[key]

# Find the installed .so and promote its site-packages to the front of sys.path.
# This handles the case where a stale editable install elsewhere on sys.path
# shadows the freshly built extension.
matches = glob.glob('/usr/local/lib/python*/dist-packages/cupyccx/_cupyccx*.so')
if matches:
    site_dir = os.path.dirname(os.path.dirname(matches[0]))
    sys.path = [site_dir] + [p for p in sys.path if p != site_dir]
    print(f'Using site-packages: {site_dir}')
else:
    print('WARNING: could not find _cupyccx*.so under /usr/local/lib — Cell 3 may not have completed')

importlib.invalidate_caches()

import cupyccx._cupyccx
print('Extension loaded from:', cupyccx._cupyccx.__file__)

In [ ]:
%%bash
# Cell 4b — install CuPy (matches Colab's CUDA 12.x runtime)
pip install -q cupy-cuda12x
python3 -c "import cupy; print('CuPy', cupy.__version__, '— CUDA', cupy.cuda.runtime.runtimeGetVersion())"

In [ ]:
# Cell 5 — imports, GPU detection, warmup
import time
import numpy as np
from pyscf import gto, scf
from cupyccx.scf_data import prepare_from_pyscf
from cupyccx.method import CCD, CCOptions
from cupyccx.py_solver import PyCCD

# Detect GPU
try:
    import subprocess
    subprocess.run(['nvidia-smi'], check=True, capture_output=True)
    USE_GPU = True
except Exception:
    USE_GPU = False
print(f'USE_GPU = {USE_GPU}')

SCALING_ITERS = 5   # fixed iterations per system (no convergence)
WALL_LIMIT_S  = 110 # skip remaining systems beyond this elapsed time
N_ATOMS       = [2, 4, 8, 16, 32, 64]  # total hydrogen atoms; doubles each step

def build_hchain(n_atoms):
    """Run RHF on a linear hydrogen chain and return SCFInputData."""
    atom = '; '.join(f'H 0 0 {i * 1.4}' for i in range(n_atoms))
    mol  = gto.M(atom=atom, basis='sto-3g', unit='Bohr', verbose=0)
    mf   = scf.RHF(mol)
    mf.verbose = 0
    mf.kernel()
    assert mf.converged, f'HF did not converge for H{n_atoms}'
    return prepare_from_pyscf(mf, verbose=False)

# Pre-build SCFInputData for all systems (shared by both sweeps)
print('Building SCF data for all systems...')
scf_data = {}
for n in N_ATOMS:
    scf_data[n] = build_hchain(n)
    d = scf_data[n]
    print(f'  H{n:<3}  n_occ={d.n_occ}  n_vir={d.n_vir}  ERI={d.n_mo**4*8/1e9:.3f} GB')

# GPU warmup: init CUDA context before timed sweeps
opts_warm = CCOptions(use_gpu=USE_GPU, max_iter=3, conv_energy=0.0, conv_amp=0.0, use_diis=False)
CCD.from_scf_data(scf_data[2], opts=opts_warm).compute(e_scf=scf_data[2].e_scf, verbose=False)
print('Warmup done.')

In [ ]:
# Cell 6 — C++ / cuBLAS backend scaling sweep
# Uses the compiled C++ extension dispatching to cuBLAS DGEMMs on GPU.
print('=== C++ / cuBLAS backend ===')
results_cpp = []
t_start = time.perf_counter()

for n_atoms in N_ATOMS:
    if time.perf_counter() - t_start > WALL_LIMIT_S:
        print(f'Wall limit reached, skipping H{n_atoms} and larger.')
        break

    data = scf_data[n_atoms]
    opts = CCOptions(use_gpu=USE_GPU, max_iter=SCALING_ITERS,
                     conv_energy=0.0, conv_amp=0.0, use_diis=False)

    t0 = time.perf_counter()
    CCD.from_scf_data(data, opts=opts).compute(e_scf=data.e_scf, verbose=False)
    dt = time.perf_counter() - t0

    print(f'  H{n_atoms:<3}  n_vir={data.n_vir:3d}  wall={dt:.3f}s')
    results_cpp.append({'label': f'H{n_atoms}', 'n_vir': data.n_vir,
                        'scale': data.n_occ**2 * data.n_vir**4, 'dt': dt})

print(f'Total: {time.perf_counter() - t_start:.1f}s')

In [ ]:
# Cell 7 — PyCCD / CuPy backend scaling sweep
# Uses the pure-Python solver with CuPy einsum contractions on GPU.
print('=== PyCCD / CuPy backend ===')
results_py = []
t_start = time.perf_counter()

for n_atoms in N_ATOMS:
    if time.perf_counter() - t_start > WALL_LIMIT_S:
        print(f'Wall limit reached, skipping H{n_atoms} and larger.')
        break

    data = scf_data[n_atoms]
    opts = CCOptions(use_gpu=USE_GPU, max_iter=SCALING_ITERS,
                     conv_energy=0.0, conv_amp=0.0, use_diis=False)

    t0 = time.perf_counter()
    PyCCD.from_scf_data(data, opts=opts).compute(e_scf=data.e_scf, verbose=False)
    dt = time.perf_counter() - t0

    print(f'  H{n_atoms:<3}  n_vir={data.n_vir:3d}  wall={dt:.3f}s')
    results_py.append({'label': f'H{n_atoms}', 'n_vir': data.n_vir,
                       'scale': data.n_occ**2 * data.n_vir**4, 'dt': dt})

print(f'Total: {time.perf_counter() - t_start:.1f}s')

In [ ]:
# Cell 8 — results table
def fmt_row(r):
    return f'{r["label"]:<8} {r["n_vir"]:>5} {r["scale"]:>12.2e} {r["dt"]:>9.3f}'

hdr = f'{"System":<8} {"n_vir":>5} {"N^6_scale":>12} {"wall(s)":>9}'
sep = '-' * 40

print('C++ / cuBLAS')
print(hdr); print(sep)
for r in results_cpp:
    print(fmt_row(r))

print()
print('PyCCD / CuPy')
print(hdr); print(sep)
for r in results_py:
    print(fmt_row(r))

In [ ]:
# Cell 9 — log-log scaling plot: C++ vs PyCCD
import matplotlib.pyplot as plt

def loglog_fit(x, y):
    c = np.polyfit(np.log10(x), np.log10(y), 1)
    return c[0], c[1]  # slope, intercept

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax_idx, (xlabel, xkey) in enumerate([
    (r'$n_{occ}^2 \cdot n_{vir}^4$ (formal CCD cost)', 'scale'),
    (r'$n_{vir}$ (virtual spin-orbitals)',               'n_vir'),
]):
    ax = axes[ax_idx]
    ideal_slope = 1.0 if xkey == 'scale' else 6.0

    for results, color, marker, name in [
        (results_cpp, 'steelblue',  'o', 'C++ / cuBLAS'),
        (results_py,  'darkorange', 's', 'PyCCD / CuPy'),
    ]:
        if not results:
            continue
        xs = np.array([r[xkey] for r in results])
        ys = np.array([r['dt'] for r in results])
        slope, ic = loglog_fit(xs, ys)

        ax.scatter(xs, ys, color=color, marker=marker, s=70, zorder=5,
                   label=f'{name}  (slope={slope:.2f})')
        for r in results:
            ax.annotate(r['label'], (r[xkey], r['dt']),
                        textcoords='offset points', xytext=(5, 2), fontsize=8,
                        color=color)
        x_fit = np.logspace(np.log10(xs.min()), np.log10(xs.max()), 200)
        ax.plot(x_fit, 10**(slope * np.log10(x_fit) + ic), '--', color=color, alpha=0.6)

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(f'Wall time / {SCALING_ITERS} iters (s)', fontsize=11)
    ax.set_title(f'ideal slope = {ideal_slope:.0f}', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, which='both', alpha=0.3)

fig.suptitle(f'CCD scaling — {"GPU" if USE_GPU else "CPU"}  |  H-chain / STO-3G',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('scaling_plot.png', dpi=150)
plt.show()

## Interpreting the results

- **Ideal slope = 1.0** on the N²_occ · N⁴_vir axis (or **6.0** on the N_vir axis)
  confirms the dominant W_vvvv contraction drives the cost as O(N⁶).
- **Slope < ideal at small sizes** (H2–H8): GPU kernel-launch overhead and
  memory-bandwidth limits dominate when matrices are tiny.
- **Slope → ideal at large sizes** (H32–H64): once DGEMMs fill the GPU's SMs,
  the empirical exponent converges to 6.
- **C++ vs PyCCD gap**: the C++ backend dispatches hand-tuned cuBLAS DGEMMs;
  PyCCD uses CuPy `einsum` which incurs higher Python overhead and intermediate
  tensor allocations — the gap widens at large N where memory traffic matters most.